# CE 477 – Data Science: Assignment #2 (Part 2: Regression)

**Institution:** Izmir University of Economics  
**Department:** Department of Computer Engineering  
**Dataset:** AFAD Seismic Activity Catalog (Preprocessed Dataset, $N = 153,439$)

---

## 1. Problem Formulation and Target Attribute Selection

In accordance with **Assignment #2 – Task 2 (Regression)**, two different numeric attributes are modelled as independent regression targets:

| Target | Physical meaning | Why it was selected |
|---|---|---|
| `magnitude` | Seismic energy release (M) | Central quantity of hazard assessment; continuous, strictly positive (MAPE well-defined). |
| `depth` | Hypocentral focal depth (km) | Separates shallow crustal ruptures from deep subduction events; independent physical dimension (Assignment #1 PCA: PC2). |

For each target, three regressors are trained on a **70% training / 30% testing** split (`random_state=42`):
1. **Linear Regression** – parametric, interpretable baseline.
2. **Random Forest Regressor** – bagged ensemble of decision trees capturing non-linear spatial patterns.
3. **HistGradientBoosting Regressor** – boosted tree ensemble.

Performance is compared using **RMSE**, **MAPE** and the **Pearson correlation coefficient (r)** (MAE and $R^2$ are reported additionally).

**Leakage control:** all target-derived columns (`*_norm`, `*_std`, `magnitude_class`, `depth_class`, `is_major`, `PCA1–3`) are excluded from the feature space. Province encodings and the `StandardScaler` are fitted on the training split only.

In [1]:
import os
os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300

RANDOM_STATE = 42
TEST_SIZE = 0.30
PLOT_SAMPLE = 6000  # points drawn in scatter plots (metrics always use the full test set)

WORKSPACE = os.path.abspath("..")
DATA_PATH = os.path.join(WORKSPACE, "data", "afad_earthquake_data_preprocessed.csv")
FIG_DIR = os.path.join(WORKSPACE, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_COLORS = {
    "Linear Regression": "#4C72B0",
    "Random Forest": "#55A868",
    "Gradient Boosting": "#C44E52",
}

## 2. Data Loading and Shared Feature Engineering
Temporal attributes are cyclically encoded ($\sin/\cos$ of month and hour) so that December–January and 23:00–00:00 are adjacent. The feature set shared by both targets contains epicentral coordinates, `rms`, `year`, the cyclic time features and the one-hot `type` columns; each target additionally uses the *other* target as a predictor (`depth` for magnitude, `magnitude` for depth).

**Depth-specific filtering:** About 19% of the catalog reports a depth of exactly **7.00 km**, which is very likely a fixed (default) solution depth assigned by the location procedure rather than an independently measured value (this interpretation should be confirmed with AFAD documentation; Section 6.1 shows how these rows are distributed over the years). These rows, together with depths below 1 km (which make MAPE numerically unstable), are removed for the depth target only.

In [2]:
print("1. Loading preprocessed dataset...")
df = pd.read_csv(DATA_PATH)
df = df.loc[:, ~df.columns.duplicated()]
print(f"Loaded dataset shape: {df.shape}")

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["month"] = df["datetime"].dt.month.fillna(6).astype(int)
df["hour"] = df["datetime"].dt.hour.fillna(12).astype(int)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

TYPE_COLS = ["type_ML", "type_Md", "type_Ml", "type_Mw", "type_Other"]
COMMON_FEATURES = [
    "latitude", "longitude", "rms", "year",
    "month_sin", "month_cos", "hour_sin", "hour_cos",
] + TYPE_COLS

# Every target-derived column (normalised / standardised copies, class labels, PCA scores)
# is excluded from the feature space to prevent target leakage.
TARGET_CONFIG = {
    "magnitude": {
        "label": "Magnitude (M)",
        "unit": "",
        "extra_features": ["depth"],
        "simple_feature": "depth",
    },
    "depth": {
        "label": "Focal Depth (km)",
        "unit": " km",
        "extra_features": ["magnitude"],
        "simple_feature": "latitude",
    },
}


def prepare_target_frame(target):
    """Return the filtered frame used for a given target."""
    data = df.copy()
    if target == "depth":
        n0 = len(data)
        # (a) depth == 7.00 km: most likely a fixed (default) solution depth, not a measurement (see Section 6.1)
        # (b) depth < 1 km makes MAPE numerically unstable (division by ~0)
        data = data[(data["depth"] != 7.0) & (data["depth"] >= 1.0)].copy()
        print(f"   Depth filter: removed {n0 - len(data):,} fixed-depth / near-zero rows "
              f"({100 * (n0 - len(data)) / n0:.2f}%) -> {len(data):,} rows")
    return data


def add_province_encoding(X_tr, X_te, y_tr):
    """Province frequency + mean-target encoding fitted on the TRAINING split only."""
    freq = X_tr["province"].value_counts(normalize=True).to_dict()
    tmp = X_tr[["province"]].copy()
    tmp["y"] = y_tr
    mean_map = tmp.groupby("province")["y"].mean().to_dict()
    global_mean = float(np.mean(y_tr))

    def _apply(part):
        out = part.drop(columns=["province"]).copy()
        out["province_freq"] = part["province"].map(freq).fillna(0.0)
        out["province_target_mean"] = part["province"].map(mean_map).fillna(global_mean)
        return out

    return _apply(X_tr), _apply(X_te)


def regression_metrics(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE (%)": float(100.0 * mean_absolute_percentage_error(y_true, y_pred)),
        "Pearson r": float(np.corrcoef(y_true, y_pred)[0, 1]),
        "R2": float(r2_score(y_true, y_pred)),
    }


results_summary = {"split": {"train": 1 - TEST_SIZE, "test": TEST_SIZE}, "targets": {}}
store = {}  # in-memory artefacts for plotting

1. Loading preprocessed dataset...


Loaded dataset shape: (153439, 36)


## 3. Model Training and Evaluation (70% / 30%)
For each target the pipeline (i) splits the data, (ii) fits province frequency / mean-target encodings on the training split, (iii) standardizes features for Linear Regression, (iv) trains the three regressors and (v) evaluates them on the held-out 30% test set. A naive **mean predictor** is included as a reference: a useful model must beat it.

In [3]:
for target, cfg in TARGET_CONFIG.items():
    print(f"\n2. Target = '{target}'")
    data = prepare_target_frame(target)
    feats = COMMON_FEATURES + cfg["extra_features"] + ["province"]
    X_raw = data[feats].copy()
    y = data[target].values.astype(float)

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"   Split -> Train: {len(X_train_raw):,} (70%), Test: {len(X_test_raw):,} (30%)")

    X_train, X_test = add_province_encoding(X_train_raw, X_test_raw, y_train)
    feature_names = list(X_train.columns)

    # Scaling is fitted on the training split only (needed for comparable LR coefficients)
    scale_cols = [c for c in feature_names if c not in TYPE_COLS]
    scaler = StandardScaler()
    X_train_s, X_test_s = X_train.copy(), X_test.copy()
    X_train_s[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_test_s[scale_cols] = scaler.transform(X_test[scale_cols])

    models = {
        "Linear Regression": (LinearRegression(), True),
        "Random Forest": (
            RandomForestRegressor(
                n_estimators=200, max_depth=20, min_samples_leaf=5,
                max_features=0.6, n_jobs=-1, random_state=RANDOM_STATE,
            ),
            False,
        ),
        "Gradient Boosting": (
            HistGradientBoostingRegressor(
                max_iter=400, learning_rate=0.08, max_leaf_nodes=63,
                min_samples_leaf=40, l2_regularization=1.0, random_state=RANDOM_STATE,
            ),
            False,
        ),
    }

    target_res = {
        "rows": int(len(data)),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "features": feature_names,
        "y_test_mean": float(np.mean(y_test)),
        "train_metrics": {},
        "test_metrics": {},
    }

    # Naive baseline: always predict the training mean
    baseline_pred = np.full_like(y_test, np.mean(y_train))
    target_res["baseline_mean_predictor"] = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, baseline_pred))),
        "MAE": float(mean_absolute_error(y_test, baseline_pred)),
        "MAPE (%)": float(100.0 * mean_absolute_percentage_error(y_test, baseline_pred)),
    }

    preds, fitted = {}, {}
    for name, (model, use_scaled) in models.items():
        print(f"   Training {name}...")
        Xtr = X_train_s if use_scaled else X_train
        Xte = X_test_s if use_scaled else X_test
        model.fit(Xtr, y_train)
        fitted[name] = model
        preds[name] = model.predict(Xte)
        target_res["train_metrics"][name] = regression_metrics(y_train, model.predict(Xtr))
        target_res["test_metrics"][name] = regression_metrics(y_test, preds[name])
        m = target_res["test_metrics"][name]
        print(f"      Test RMSE={m['RMSE']:.4f}  MAPE={m['MAPE (%)']:.2f}%  "
              f"r={m['Pearson r']:.4f}  R2={m['R2']:.4f}")

    lr = fitted["Linear Regression"]
    target_res["linear_coefficients_standardized"] = dict(
        sorted(zip(feature_names, map(float, lr.coef_)), key=lambda kv: -abs(kv[1]))
    )
    target_res["linear_intercept"] = float(lr.intercept_)
    target_res["rf_feature_importance"] = dict(
        sorted(zip(feature_names, map(float, fitted["Random Forest"].feature_importances_)),
               key=lambda kv: -kv[1])
    )

    # Simple (univariate) least-squares line for the "scatter + regression line" view
    sf = cfg["simple_feature"]
    slope, intercept = np.polyfit(X_train_raw[sf].values, y_train, 1)
    simple_pred = slope * X_test_raw[sf].values + intercept
    target_res["simple_linear"] = {
        "feature": sf,
        "slope": float(slope),
        "intercept": float(intercept),
        "test_metrics": regression_metrics(y_test, simple_pred),
    }

    results_summary["targets"][target] = target_res
    store[target] = {
        "y_test": y_test,
        "preds": preds,
        "X_train_raw": X_train_raw,
        "X_test_raw": X_test_raw,
        "y_train": y_train,
        "simple": (sf, slope, intercept),
    }


2. Target = 'magnitude'
   Split -> Train: 107,407 (70%), Test: 46,032 (30%)
   Training Linear Regression...
      Test RMSE=0.3398  MAPE=8.18%  r=0.5791  R2=0.3351
   Training Random Forest...


      Test RMSE=0.2944  MAPE=7.05%  r=0.7078  R2=0.5008
   Training Gradient Boosting...


      Test RMSE=0.2942  MAPE=7.04%  r=0.7083  R2=0.5016

2. Target = 'depth'
   Depth filter: removed 29,536 fixed-depth / near-zero rows (19.25%) -> 123,903 rows
   Split -> Train: 86,732 (70%), Test: 37,171 (30%)
   Training Linear Regression...
      Test RMSE=8.3584  MAPE=57.10%  r=0.3617  R2=0.1307
   Training Random Forest...


      Test RMSE=7.5323  MAPE=50.01%  r=0.5438  R2=0.2941
   Training Gradient Boosting...


      Test RMSE=7.5147  MAPE=50.74%  r=0.5466  R2=0.2973


## 4. Visualization of the Learned Models
* **Fig. 18 / Fig. 19:** actual vs. predicted scatter plots with the fitted least-squares regression line (black) and the ideal $\hat{y}=y$ line (dashed).
* **Fig. 20:** simple (univariate) linear regression – scatter plot of the training data with the fitted OLS line.
* **Fig. 21:** RMSE, MAPE and correlation comparison against the mean baseline.
* **Fig. 22:** standardized Linear Regression coefficients and Random Forest feature importances.

In [4]:
rng = np.random.default_rng(RANDOM_STATE)


def actual_vs_predicted_figure(target, fig_num):
    cfg = TARGET_CONFIG[target]
    s = store[target]
    y_test = s["y_test"]
    idx = rng.choice(len(y_test), size=min(PLOT_SAMPLE, len(y_test)), replace=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharex=True, sharey=True)
    lo, hi = np.percentile(y_test, [0.2, 99.8])
    pad = 0.05 * (hi - lo)
    lim = (lo - pad, hi + pad)
    for ax, (name, y_pred) in zip(axes, s["preds"].items()):
        ax.scatter(y_test[idx], y_pred[idx], s=7, alpha=0.30,
                   color=MODEL_COLORS[name], edgecolors="none", label="Test samples")
        # Least-squares regression line of predicted on actual (full test set)
        a, b = np.polyfit(y_test, y_pred, 1)
        xs = np.linspace(*lim, 100)
        ax.plot(xs, a * xs + b, color="black", lw=2.0, label=f"Fit: ŷ = {a:.2f}·y + {b:.2f}")
        ax.plot(lim, lim, ls="--", color="grey", lw=1.3, label="Ideal (ŷ = y)")
        m = results_summary["targets"][target]["test_metrics"][name]
        ax.set_title(f"{name}\nRMSE={m['RMSE']:.3f}{cfg['unit']} | "
                     f"MAPE={m['MAPE (%)']:.1f}% | r={m['Pearson r']:.3f}", fontsize=11)
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_xlabel(f"Actual {cfg['label']}")
        ax.legend(loc="upper left", fontsize=8.5, frameon=True)
    axes[0].set_ylabel(f"Predicted {cfg['label']}")
    fig.suptitle(f"Actual vs. Predicted {cfg['label']} on the 30% Test Set "
                 f"(n = {len(y_test):,}; {len(idx):,} points shown)", fontsize=13, y=1.02)
    path = os.path.join(FIG_DIR, f"fig{fig_num}_regression_{target}_actual_vs_predicted.png")
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"   Saved {os.path.basename(path)}")


print("\n3. Generating figures...")
actual_vs_predicted_figure("magnitude", 18)
actual_vs_predicted_figure("depth", 19)


3. Generating figures...


   Saved fig18_regression_magnitude_actual_vs_predicted.png


   Saved fig19_regression_depth_actual_vs_predicted.png


In [5]:
# Figure 20: simple linear regression lines (scatter with regression line on a single predictor)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
for ax, target in zip(axes, ["magnitude", "depth"]):
    cfg = TARGET_CONFIG[target]
    s = store[target]
    sf, slope, intercept = s["simple"]
    Xtr, ytr = s["X_train_raw"][sf].values, s["y_train"]
    idx = rng.choice(len(ytr), size=min(PLOT_SAMPLE, len(ytr)), replace=False)
    ax.scatter(Xtr[idx], ytr[idx], s=7, alpha=0.25, color="#4C72B0", edgecolors="none",
               label="Training samples")
    xs = np.linspace(np.percentile(Xtr, 0.5), np.percentile(Xtr, 99.5), 100)
    ax.plot(xs, slope * xs + intercept, color="#C44E52", lw=2.5,
            label=f"OLS: y = {slope:.4f}·x + {intercept:.3f}")
    sm = results_summary["targets"][target]["simple_linear"]["test_metrics"]
    ax.set_title(f"{cfg['label']} ~ {sf}\nTest RMSE={sm['RMSE']:.3f}{cfg['unit']} | "
                 f"r={sm['Pearson r']:.3f}", fontsize=11)
    ax.set_xlabel(sf.capitalize() + (" (km)" if sf == "depth" else " (°N)" if sf == "latitude" else ""))
    ax.set_ylabel(cfg["label"])
    ax.set_ylim(np.percentile(ytr, 0.2), np.percentile(ytr, 99.8))
    ax.set_xlim(xs[0], xs[-1])
    ax.legend(loc="upper right", fontsize=9)
fig.suptitle("Simple Linear Regression: Scatter Plot with Fitted Regression Line", fontsize=13, y=1.02)
path = os.path.join(FIG_DIR, "fig20_simple_linear_regression_lines.png")
plt.savefig(path, bbox_inches="tight")
plt.show()
print(f"   Saved {os.path.basename(path)}")

   Saved fig20_simple_linear_regression_lines.png


In [6]:
# Figure 21: metric comparison (RMSE, MAPE, Pearson r) for both targets
metrics_to_plot = [("RMSE", "RMSE (lower is better)"),
                   ("MAPE (%)", "MAPE % (lower is better)"),
                   ("Pearson r", "Correlation r (higher is better)")]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for row, target in enumerate(["magnitude", "depth"]):
    tm = results_summary["targets"][target]["test_metrics"]
    names = list(tm.keys())
    for col, (mkey, mtitle) in enumerate(metrics_to_plot):
        ax = axes[row, col]
        vals = [tm[n][mkey] for n in names]
        bars = ax.bar(names, vals, color=[MODEL_COLORS[n] for n in names], edgecolor="black", lw=0.6)
        for b_, v in zip(bars, vals):
            ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9.5, fontweight="bold")
        if mkey in ("RMSE", "MAPE (%)"):
            base = results_summary["targets"][target]["baseline_mean_predictor"][mkey]
            ax.axhline(base, color="black", ls=":", lw=1.4, label=f"Mean baseline = {base:.3f}")
            ax.legend(fontsize=8.5, loc="lower right")
        if mkey == "Pearson r":
            ax.set_ylim(0, 1)
        else:
            ax.set_ylim(0, max(vals + [results_summary["targets"][target]["baseline_mean_predictor"][mkey]]) * 1.18)
        ax.set_title(f"{TARGET_CONFIG[target]['label']} — {mtitle}", fontsize=10.5)
        ax.tick_params(axis="x", labelsize=9)
fig.suptitle("Regression Performance Comparison on the 30% Test Set", fontsize=14, y=1.01)
plt.tight_layout()
path = os.path.join(FIG_DIR, "fig21_regression_metrics_comparison.png")
plt.savefig(path, bbox_inches="tight")
plt.show()
print(f"   Saved {os.path.basename(path)}")

   Saved fig21_regression_metrics_comparison.png


In [7]:
# Figure 22: learned model internals — LR standardized coefficients & RF feature importance
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for row, target in enumerate(["magnitude", "depth"]):
    tr = results_summary["targets"][target]
    coefs = pd.Series(tr["linear_coefficients_standardized"]).head(10)[::-1]
    axes[row, 0].barh(coefs.index, coefs.values,
                      color=["#C44E52" if v < 0 else "#4C72B0" for v in coefs.values], edgecolor="black", lw=0.5)
    axes[row, 0].axvline(0, color="black", lw=0.8)
    axes[row, 0].set_title(f"{TARGET_CONFIG[target]['label']} — Linear Regression coefficients\n"
                           "(standardized features, top 10 by |β|)", fontsize=10.5)
    imp = pd.Series(tr["rf_feature_importance"]).head(10)[::-1]
    axes[row, 1].barh(imp.index, imp.values, color="#55A868", edgecolor="black", lw=0.5)
    axes[row, 1].set_title(f"{TARGET_CONFIG[target]['label']} — Random Forest feature importance\n"
                           "(impurity decrease, top 10)", fontsize=10.5)
plt.tight_layout()
path = os.path.join(FIG_DIR, "fig22_regression_coefficients_and_importance.png")
plt.savefig(path, bbox_inches="tight")
plt.show()
print(f"   Saved {os.path.basename(path)}")

   Saved fig22_regression_coefficients_and_importance.png


## 5. Results Summary

In [8]:
print("\n4. Test-set summary")
rows = []
for target, tr in results_summary["targets"].items():
    for name, m in tr["test_metrics"].items():
        rows.append({"Target": target, "Model": name, **{k: round(v, 4) for k, v in m.items()}})
    b = tr["baseline_mean_predictor"]
    rows.append({"Target": target, "Model": "Mean baseline",
                 "RMSE": round(b["RMSE"], 4), "MAE": round(b["MAE"], 4), "MAPE (%)": round(b["MAPE (%)"], 4)})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

print("\nTrain vs. test RMSE (overfitting check)")
for target, tr in results_summary["targets"].items():
    for name in tr["test_metrics"]:
        print(f"  {target:9s} {name:18s} train={tr['train_metrics'][name]['RMSE']:.4f} "
              f"test={tr['test_metrics'][name]['RMSE']:.4f}")

print("\nResults JSON:")
print(json.dumps({t: {"test_metrics": v["test_metrics"],
                      "simple_linear": v["simple_linear"],
                      "baseline_mean_predictor": v["baseline_mean_predictor"]}
                  for t, v in results_summary["targets"].items()}, indent=2))
print("\nDone.")


4. Test-set summary
   Target             Model   RMSE    MAE  MAPE (%)  Pearson r     R2
magnitude Linear Regression 0.3398 0.2465    8.1797     0.5791 0.3351
magnitude     Random Forest 0.2944 0.2120    7.0480     0.7078 0.5008
magnitude Gradient Boosting 0.2942 0.2117    7.0358     0.7083 0.5016
magnitude     Mean baseline 0.4167 0.3001    9.8376        NaN    NaN
    depth Linear Regression 8.3584 4.4316   57.0993     0.3617 0.1307
    depth     Random Forest 7.5323 3.8690   50.0103     0.5438 0.2941
    depth Gradient Boosting 7.5147 3.9204   50.7440     0.5466 0.2973
    depth     Mean baseline 8.9649 4.8732   62.6799        NaN    NaN

Train vs. test RMSE (overfitting check)
  magnitude Linear Regression  train=0.3382 test=0.3398
  magnitude Random Forest      train=0.2309 test=0.2944
  magnitude Gradient Boosting  train=0.2719 test=0.2942
  depth     Linear Regression  train=8.1129 test=8.3584
  depth     Random Forest      train=5.6382 test=7.5323
  depth     Gradient Boostin

## 6. Additional Experiments (Diagnostics and Robustness Checks)

The experiments below do **not** change the main results of Sections 3–5. They (i) compute every data fact that the discussion relies on, (ii) justify the choice of predictor in the simple regression plot (Fig. 20), (iii) test how much each feature group contributes (ablation), (iv) inspect the residuals and (v) quantify whether the differences between models are larger than sampling noise. All experiments re-use the same 70% / 30% split; models are only *diagnosed*, never re-selected, on the test set.

| Section | Question answered |
|---|---|
| 6.1 | Data facts used in the discussion (depth distribution, 7.00 km rows, magnitude-type composition) |
| 6.2 | Which single predictor is most correlated with each target? (justifies Fig. 20) |
| 6.3 | Ablation: other target removed, `type_*` removed, province encodings removed / out-of-fold, log-depth target, Ridge, no depth filter; training times |
| 6.4 | Residual analysis and error by target range (Fig. 27) |
| 6.5 | Paired bootstrap: are the RMSE differences between models significant? |

In [9]:
# 6.1 Data facts referenced in the discussion (computed, not assumed)
os.makedirs(os.path.join(WORKSPACE, "results"), exist_ok=True)
RES_DIR = os.path.join(WORKSPACE, "results")
summary_df.to_csv(os.path.join(RES_DIR, "reg_main_test_summary.csv"), index=False)

d_all = df["depth"]
d_flt = prepare_target_frame("depth")["depth"]
facts = pd.DataFrame({
    "Subset": ["All rows", "Depth target (after filter)"],
    "n": [len(d_all), len(d_flt)],
    "Median depth (km)": [d_all.median(), d_flt.median()],
    "Q1 (km)": [d_all.quantile(0.25), d_flt.quantile(0.25)],
    "Q3 (km)": [d_all.quantile(0.75), d_flt.quantile(0.75)],
    "Share depth == 7.00 km (%)": [100 * (d_all == 7.0).mean(), 100 * (d_flt == 7.0).mean()],
    "Share depth < 1 km (%)": [100 * (d_all < 1.0).mean(), 100 * (d_flt < 1.0).mean()],
})
print("=== Depth distribution ===")
print(facts.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\n=== Rows with depth exactly 7.00 km, by year (last 10 years) ===")
by_year = df.groupby("year")["depth"].apply(lambda s: pd.Series({"n_7km": int((s == 7.0).sum()), "share_%": 100 * (s == 7.0).mean()})).unstack()
print(by_year.tail(10).to_string(float_format=lambda x: f"{x:.1f}"))

print("\n=== Composition of the one-hot group type_Other (raw `type` labels) ===")
print(df.loc[df["type_Other"] == 1, "type"].value_counts().to_string())

mag_by_type = df.groupby("type")["magnitude"].agg(["count", "mean", "median"]).sort_values("count", ascending=False)
print("\n=== Magnitude by raw magnitude type ===")
print(mag_by_type.to_string(float_format=lambda x: f"{x:.3f}"))
facts.to_csv(os.path.join(RES_DIR, "reg_data_facts_depth.csv"), index=False)
mag_by_type.to_csv(os.path.join(RES_DIR, "reg_magnitude_by_type.csv"))

   Depth filter: removed 29,536 fixed-depth / near-zero rows (19.25%) -> 123,903 rows
=== Depth distribution ===
                     Subset      n  Median depth (km)  Q1 (km)  Q3 (km)  Share depth == 7.00 km (%)  Share depth < 1 km (%)
                   All rows 153439               7.00     6.96     8.71                       19.20                    0.05
Depth target (after filter) 123903               7.04     6.89    10.40                        0.00                    0.00

=== Rows with depth exactly 7.00 km, by year (last 10 years) ===
      n_7km  share_%
year                
2017  425.0     11.0
2018  188.0      9.7
2019  377.0     16.2
2020  967.0     20.8
2021  296.0     10.1
2022  315.0     15.5
2023 3423.0     30.2
2024  643.0     22.3
2025 2238.0     38.7
2026  407.0     20.6

=== Composition of the one-hot group type_Other (raw `type` labels) ===
type
MW        2424
Mwp          2
Ms(BB)       1

=== Magnitude by raw magnitude type ===
        count  mean  median
type 

In [10]:
# 6.2 Justification of the predictor used in the simple regression plot (Fig. 20)
# Pearson correlation of every candidate predictor with the target, computed on the TRAINING split only.
ENGINEERED = ["province_freq", "province_target_mean"]   # derived from the training targets / counts, not raw attributes
corr_rows = []
for target in ["magnitude", "depth"]:
    s = store[target]
    Xtr_enc, _ = add_province_encoding(s["X_train_raw"], s["X_test_raw"], s["y_train"])
    for c in Xtr_enc.columns:
        r = np.corrcoef(Xtr_enc[c].astype(float), s["y_train"])[0, 1]
        kind = "engineered (target/frequency encoding)" if c in ENGINEERED else ("binary one-hot" if Xtr_enc[c].nunique() <= 2 else "raw continuous")
        corr_rows.append({"Target": target, "Predictor": c, "Kind": kind, "r": r, "|r|": abs(r)})
corr_df = pd.DataFrame(corr_rows).sort_values(["Target", "|r|"], ascending=[False, False])
for target in ["magnitude", "depth"]:
    t = corr_df[corr_df["Target"] == target]
    print(f"=== {target}: correlation with the target (training split, top 8) ===")
    print(t.head(8).drop(columns="Target").to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    best_raw = t[t["Kind"] == "raw continuous"].iloc[0]
    print(f"-> strongest RAW continuous predictor: {best_raw['Predictor']} (r = {best_raw['r']:.3f}); "
          f"predictor used in Fig. 20: {TARGET_CONFIG[target]['simple_feature']}\n")
corr_df.to_csv(os.path.join(RES_DIR, "reg_univariate_correlations.csv"), index=False)

=== magnitude: correlation with the target (training split, top 8) ===
           Predictor                                   Kind      r   |r|
             type_Mw                         binary one-hot  0.385 0.385
          type_Other                         binary one-hot  0.349 0.349
province_target_mean engineered (target/frequency encoding)  0.196 0.196
               depth                         raw continuous  0.182 0.182
             type_Md                         binary one-hot -0.138 0.138
            latitude                         raw continuous -0.107 0.107
             type_ML                         binary one-hot -0.090 0.090
                year                         raw continuous  0.061 0.061
-> strongest RAW continuous predictor: depth (r = 0.182); predictor used in Fig. 20: depth

=== depth: correlation with the target (training split, top 8) ===
           Predictor                                   Kind      r   |r|
province_target_mean engineered (target/

In [11]:
# 6.3 Ablation study and training times (Gradient Boosting unless stated otherwise; same 70/30 split)
import time
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold


def make_gbr():
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, max_leaf_nodes=63,
                                         min_samples_leaf=40, l2_regularization=1.0, random_state=RANDOM_STATE)


def oof_province_mean(prov_tr, y_tr, n_splits=5):
    prov_tr = prov_tr.reset_index(drop=True)
    out, gm = np.zeros(len(prov_tr)), float(np.mean(y_tr))
    for a, b in KFold(n_splits, shuffle=True, random_state=RANDOM_STATE).split(prov_tr):
        mp = pd.Series(y_tr[a]).groupby(prov_tr.iloc[a].values).mean()
        out[b] = prov_tr.iloc[b].map(mp).fillna(gm).values
    return out


abl_rows, time_rows = [], []
for target, cfg in TARGET_CONFIG.items():
    s = store[target]
    ytr, yte = s["y_train"], s["y_test"]
    Xtr, Xte = add_province_encoding(s["X_train_raw"], s["X_test_raw"], ytr)
    other = cfg["extra_features"][0]

    def run(label, Xa, Xb, model=None, y_transform=None):
        mdl = model if model is not None else make_gbr()
        yfit = ytr if y_transform is None else y_transform[0](ytr)
        t0 = time.perf_counter()
        mdl.fit(Xa, yfit)
        secs = time.perf_counter() - t0
        p = mdl.predict(Xb)
        if y_transform is not None:
            p = y_transform[1](p)
        abl_rows.append({"Target": target, "Scenario": label, "n features": Xa.shape[1], **regression_metrics(yte, p)})
        return secs

    run("Reference (all features)", Xtr, Xte)
    run(f"Without the other target ('{other}')", Xtr.drop(columns=[other]), Xte.drop(columns=[other]))
    run("Without type_* one-hot columns", Xtr.drop(columns=TYPE_COLS), Xte.drop(columns=TYPE_COLS))
    run("Without province encodings", Xtr.drop(columns=["province_freq", "province_target_mean"]),
        Xte.drop(columns=["province_freq", "province_target_mean"]))
    Xtr_oof = Xtr.copy()
    Xtr_oof["province_target_mean"] = oof_province_mean(s["X_train_raw"]["province"], ytr)
    run("Out-of-fold province target encoding", Xtr_oof, Xte)
    if target == "depth":
        run("log1p(depth) target, back-transformed", Xtr, Xte, y_transform=(np.log1p, np.expm1))
    # Regularized linear model (alpha chosen by internal CV on the training split)
    scale_cols = [c for c in Xtr.columns if c not in TYPE_COLS]
    sc = StandardScaler().fit(Xtr[scale_cols])
    Xtr_s, Xte_s = Xtr.copy(), Xte.copy()
    Xtr_s[scale_cols], Xte_s[scale_cols] = sc.transform(Xtr[scale_cols]), sc.transform(Xte[scale_cols])
    ridge = RidgeCV(alphas=np.logspace(-3, 3, 13))
    run("Ridge regression (RidgeCV, linear)", Xtr_s, Xte_s, model=ridge)
    print(f"{target}: RidgeCV selected alpha = {ridge.alpha_:.4g}")

    # Training time of the three main models (single run on this machine; indicative only)
    for name, (mdl, scaled) in {
        "Linear Regression": (LinearRegression(), True),
        "Random Forest": (RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5, max_features=0.6,
                                                n_jobs=-1, random_state=RANDOM_STATE), False),
        "Gradient Boosting": (make_gbr(), False),
    }.items():
        t0 = time.perf_counter()
        mdl.fit(Xtr_s if scaled else Xtr, ytr)
        time_rows.append({"Target": target, "Model": name, "Fit time (s)": time.perf_counter() - t0})

# Depth without the 7.00 km / <1 km filter (different rows, so a different test set; MAPE is undefined because of 0 km depths)
data_all = df.copy()
X_raw_all = data_all[COMMON_FEATURES + ["magnitude", "province"]]
y_all = data_all["depth"].values.astype(float)
Xa_raw, Xb_raw, ya, yb = train_test_split(X_raw_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)
Xa, Xb = add_province_encoding(Xa_raw, Xb_raw, ya)
p_all = make_gbr().fit(Xa, ya).predict(Xb)
base_all = np.full_like(yb, ya.mean())
m_all = regression_metrics(yb, p_all); m_all["MAPE (%)"] = np.nan   # undefined: the unfiltered test set contains depth = 0 km
abl_rows.append({"Target": "depth", "Scenario": "No depth filter (all rows, own 70/30 split)", "n features": Xa.shape[1], **m_all})
abl_rows.append({"Target": "depth", "Scenario": "No depth filter: mean baseline", "n features": 0,
                 "RMSE": float(np.sqrt(mean_squared_error(yb, base_all))), "MAE": float(mean_absolute_error(yb, base_all)),
                 "MAPE (%)": np.nan, "Pearson r": np.nan, "R2": float(r2_score(yb, base_all))})
mask7 = Xb_raw.index.isin(data_all.index[data_all["depth"] == 7.0])
print(f"\nNo-filter model: share of test predictions within ±0.5 km of 7.00 km = {100 * np.mean(np.abs(p_all - 7.0) < 0.5):.1f}% "
      f"(true share of 7.00 km rows in that test set = {100 * mask7.mean():.1f}%)")

abl_df = pd.DataFrame(abl_rows)
print("\n=== Ablation (test set, Gradient Boosting unless stated) ===")
print(abl_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
time_df = pd.DataFrame(time_rows)
print("\n=== Training time (seconds, one run) ===")
print(time_df.pivot(index="Model", columns="Target", values="Fit time (s)").to_string(float_format=lambda x: f"{x:.2f}"))
abl_df.to_csv(os.path.join(RES_DIR, "reg_ablation_test.csv"), index=False)
time_df.to_csv(os.path.join(RES_DIR, "reg_training_times.csv"), index=False)

magnitude: RidgeCV selected alpha = 0.3162


depth: RidgeCV selected alpha = 31.62



No-filter model: share of test predictions within ±0.5 km of 7.00 km = 23.4% (true share of 7.00 km rows in that test set = 19.0%)

=== Ablation (test set, Gradient Boosting unless stated) ===
   Target                                    Scenario  n features   RMSE    MAE  MAPE (%)  Pearson r      R2
magnitude                    Reference (all features)          16 0.2942 0.2117    7.0358     0.7083  0.5016
magnitude          Without the other target ('depth')          15 0.3062 0.2203    7.3143     0.6785  0.4603
magnitude              Without type_* one-hot columns          11 0.3628 0.2528    8.3091     0.4920  0.2420
magnitude                  Without province encodings          14 0.2942 0.2118    7.0410     0.7082  0.5015
magnitude        Out-of-fold province target encoding          16 0.2946 0.2120    7.0420     0.7074  0.5003
magnitude          Ridge regression (RidgeCV, linear)          16 0.3398 0.2465    8.1797     0.5791  0.3351
    depth                    Reference (all

In [12]:
# 6.4 Residual analysis of Gradient Boosting (test set) and error by target range
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
range_rows = []
for row, target in enumerate(["magnitude", "depth"]):
    cfg = TARGET_CONFIG[target]
    s = store[target]
    y_true, y_pred = s["y_test"], s["preds"]["Gradient Boosting"]
    resid = y_true - y_pred
    idx = rng.choice(len(y_true), size=min(PLOT_SAMPLE, len(y_true)), replace=False)
    ax = axes[row, 0]
    ax.scatter(y_pred[idx], resid[idx], s=6, alpha=0.3, color="#C44E52", edgecolors="none")
    ax.axhline(0, color="black", lw=1)
    ax.set_xlabel(f"Predicted {cfg['label']}")
    ax.set_ylabel("Residual (actual − predicted)")
    ax.set_ylim(np.percentile(resid, [0.3, 99.7]))
    ax.set_title(f"{cfg['label']}: residuals vs. predictions (Gradient Boosting)", fontsize=11)
    ax = axes[row, 1]
    ax.hist(np.clip(resid, *np.percentile(resid, [0.5, 99.5])), bins=60, color="#4C72B0", edgecolor="white")
    ax.axvline(0, color="black", lw=1)
    ax.set_title(f"{cfg['label']}: residual distribution (mean = {resid.mean():+.3f}, skew = {pd.Series(resid).skew():.2f})", fontsize=11)
    ax.set_xlabel("Residual")

    bins = np.unique(np.quantile(y_true, [0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))
    labels = pd.cut(y_true, bins=bins, include_lowest=True)
    for interval, grp in pd.DataFrame({"y": y_true, "p": y_pred, "bin": labels}).groupby("bin", observed=True):
        e = grp["y"] - grp["p"]
        range_rows.append({"Target": target, "Actual range": str(interval), "n": len(grp),
                           "Mean actual": grp["y"].mean(), "Mean predicted": grp["p"].mean(),
                           "Bias (actual − pred)": e.mean(), "RMSE": np.sqrt((e ** 2).mean())})
plt.suptitle("Residual Analysis on the 30% Test Set", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig27_regression_residuals.png"), bbox_inches="tight")
plt.show()
range_df = pd.DataFrame(range_rows)
print("Slope of the least-squares line of predicted on actual values (1.0 = no shrinkage toward the mean):")
for target in ["magnitude", "depth"]:
    for name, yp in store[target]["preds"].items():
        print(f"  {target:9s} {name:18s} slope = {np.polyfit(store[target]['y_test'], yp, 1)[0]:.3f}")
print()
print(range_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
range_df.to_csv(os.path.join(RES_DIR, "reg_error_by_target_range.csv"), index=False)

Slope of the least-squares line of predicted on actual values (1.0 = no shrinkage toward the mean):
  magnitude Linear Regression  slope = 0.328
  magnitude Random Forest      slope = 0.492
  magnitude Gradient Boosting  slope = 0.498
  depth     Linear Regression  slope = 0.129
  depth     Random Forest      slope = 0.274
  depth     Gradient Boosting  slope = 0.278

   Target    Actual range     n  Mean actual  Mean predicted  Bias (actual − pred)   RMSE
magnitude    (2.499, 2.6] 13264        2.554           2.775                -0.221  0.253
magnitude      (2.6, 2.8] 12895        2.747           2.813                -0.065  0.149
magnitude      (2.8, 3.1] 10829        2.981           2.895                 0.086  0.197
magnitude      (3.1, 3.4]  4443        3.281           3.035                 0.246  0.363
magnitude      (3.4, 4.4]  4187        3.785           3.457                 0.328  0.533
magnitude      (4.4, 7.2]   414        4.874           3.751                 1.123  1.254

In [13]:
# 6.5 Paired bootstrap (1,000 resamples of the test set) for RMSE differences between models
B = 1000
rng_b = np.random.default_rng(RANDOM_STATE)
boot_rows = []
for target in ["magnitude", "depth"]:
    s = store[target]
    y_true = s["y_test"]
    se = {k: (y_true - v) ** 2 for k, v in s["preds"].items()}
    n = len(y_true)
    draws = {k: np.empty(B) for k in se}
    for b in range(B):
        idx = rng_b.integers(0, n, n)
        for k in se:
            draws[k][b] = np.sqrt(se[k][idx].mean())
    for a_, b_ in [("Gradient Boosting", "Random Forest"), ("Gradient Boosting", "Linear Regression"), ("Random Forest", "Linear Regression")]:
        d = draws[a_] - draws[b_]
        lo, hi = np.percentile(d, [2.5, 97.5])
        boot_rows.append({"Target": target, "Comparison": f"RMSE({a_}) − RMSE({b_})",
                          "Point": np.sqrt(se[a_].mean()) - np.sqrt(se[b_].mean()), "CI 2.5%": lo, "CI 97.5%": hi,
                          "Significant (CI excludes 0)": bool(lo > 0 or hi < 0)})
boot_df = pd.DataFrame(boot_rows)
print(boot_df.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
boot_df.to_csv(os.path.join(RES_DIR, "reg_bootstrap_rmse_differences.csv"), index=False)

   Target                                        Comparison   Point  CI 2.5%  CI 97.5%  Significant (CI excludes 0)
magnitude     RMSE(Gradient Boosting) − RMSE(Random Forest) -0.0002  -0.0009   +0.0004                        False
magnitude RMSE(Gradient Boosting) − RMSE(Linear Regression) -0.0456  -0.0477   -0.0434                         True
magnitude     RMSE(Random Forest) − RMSE(Linear Regression) -0.0454  -0.0475   -0.0432                         True
    depth     RMSE(Gradient Boosting) − RMSE(Random Forest) -0.0176  -0.0709   +0.0344                        False
    depth RMSE(Gradient Boosting) − RMSE(Linear Regression) -0.8437  -0.9426   -0.7408                         True
    depth     RMSE(Random Forest) − RMSE(Linear Regression) -0.8261  -0.9249   -0.7246                         True


## 7. Discussion

**Test-set results (30% hold-out; values printed in Section 5):**

| Target | Model | RMSE | MAPE (%) | Pearson r | $R^2$ |
|---|---|---|---|---|---|
| Magnitude | Mean baseline | 0.417 | 9.84 | – | – |
| Magnitude | Linear Regression | 0.340 | 8.18 | 0.579 | 0.335 |
| Magnitude | Random Forest | 0.294 | 7.05 | 0.708 | 0.501 |
| Magnitude | Gradient Boosting | **0.294** | **7.04** | **0.708** | **0.502** |
| Depth (km) | Mean baseline | 8.965 | 62.68 | – | – |
| Depth (km) | Linear Regression | 8.358 | 57.10 | 0.362 | 0.131 |
| Depth (km) | Random Forest | 7.532 | **50.01** | 0.544 | 0.294 |
| Depth (km) | Gradient Boosting | **7.515** | 50.74 | **0.547** | **0.297** |

**1. Non-linear ensembles clearly outperform the linear model, but are tied with each other.** For both targets, Random Forest and Gradient Boosting reduce RMSE by about 13% (magnitude, 0.340 → 0.294) and 10% (depth, 8.36 → 7.51 km) compared to Linear Regression, and raise the correlation from 0.58 → 0.71 and 0.36 → 0.55. The paired bootstrap in Section 6.5 confirms that these improvements are not sampling noise: the 95% interval of RMSE(GB) − RMSE(LR) is [−0.048, −0.043] for magnitude and [−0.94, −0.74] km for depth. The difference between the two ensembles, however, is *not* significant (magnitude: [−0.0009, +0.0004]; depth: [−0.071, +0.034] km). The remaining error is therefore caused by the information content of the features rather than by the choice of tree ensemble. A regularised linear model (RidgeCV, Section 6.3) gives exactly the same result as ordinary Linear Regression, so the linear model is limited by its functional form, not by overfitting.

**2. Why the linear model is limited.** Seismic behaviour depends on *where* an event happens (fault zone, subduction slab) in a strongly non-linear way. A single linear plane over latitude/longitude cannot represent such localised patterns, whereas trees split the map into regions. The univariate fits in Fig. 20 show how little a single raw attribute explains: depth alone explains little of magnitude ($r = 0.18$) and latitude alone little of depth ($r = -0.20$). Section 6.2 shows that these are the *strongest* raw continuous predictors for each target (computed on the training split), so the weak univariate fit is a property of the data and not a poor choice of predictor. Stronger single correlations exist only for one-hot magnitude-type columns (`type_Mw`, $r = 0.385$) and for the engineered province mean of the target ($r = 0.32$ for depth).

**3. Magnitude is predictable mainly through the measurement method.** Both the Linear Regression coefficients and the Random Forest importances (Fig. 22) are dominated by the `type` one-hot columns. Section 6.1 shows why: moment magnitudes are reported mainly for larger events (mean M = 3.80 for `Mw` and 4.06 for `MW`, versus 2.85–2.91 for `Md`, `ML` and `Ml`), and the `type_Other` group consists almost entirely of upper-case `MW` labels (2,424 of 2,427 rows). The ablation in Section 6.3 quantifies the effect: removing the five `type_*` columns lowers the magnitude $R^2$ from 0.50 to 0.24 and the correlation from 0.71 to 0.49, while it has no effect on depth. This is a catalogue-procedure effect, not a physical precursor, so the model should be interpreted as *estimating* magnitude from catalogue metadata, not as *forecasting* earthquakes.

**4. The two targets partly explain each other.** Each target uses the other one as an input. Removing it (Section 6.3) lowers $R^2$ from 0.50 to 0.46 for magnitude and from 0.30 to 0.27 for depth. The contribution is real but modest, so the results do not depend on this choice. Province encodings contribute almost nothing: removing them leaves the magnitude score unchanged and even slightly improves depth (RMSE 7.51 → 7.46 km), and an out-of-fold version of the province target encoding gives the same scores as the in-sample version (difference ≤ 0.002 in $R^2$). The in-sample target encoding therefore does not inflate the reported test results.

**5. Regression toward the mean.** In Fig. 18 and Fig. 19 the fitted lines are flatter than the ideal $\hat{y}=y$ line (slope 0.49–0.50 for the ensembles on magnitude and 0.27–0.28 on depth; Section 6.4). The error-by-range table in Section 6.4 shows the consequence: events with M > 4.4 are under-estimated by 1.12 on average, and the deepest 1% of events (> 47.6 km) by 42.6 km, while the smallest magnitudes and shallowest depths are over-estimated. Rare large/deep events are under-represented (Gutenberg–Richter distribution), and squared-error training favours the dense region around M ≈ 2.5–3.0 and depth ≈ 5–15 km. This is the regression counterpart of the class imbalance in the classification task.

**6. Depth is the harder target, and MAPE exaggerates the difficulty.** Depth MAPE is around 50% while RMSE is around 7.5 km. Most depths are small (after filtering: median 7.04 km, IQR 6.89–10.40 km; Section 6.1), so a 4 km error on a 3 km event becomes a > 100% percentage error. Training on $\log(1+\text{depth})$ illustrates the metric trade-off: MAPE drops from 50.7% to 39.0% and MAE from 3.92 to 3.60 km, but RMSE rises from 7.51 to 7.85 km because deep events are compressed. RMSE and correlation are therefore more meaningful for depth than MAPE. For magnitude the opposite caution applies: MAPE is low (≈ 7%) largely because the catalogue only contains M ≥ 2.5, so even the mean baseline reaches 9.8%.

**7. Effect of the depth filter.** About 19.2% of all rows have a depth of exactly 7.00 km (Section 6.1), and this share varies strongly by year (e.g. 30% in 2023 and 39% in 2025), which supports treating them as a fixed solution depth rather than a measurement (to be confirmed with AFAD documentation). Training on all rows instead (Section 6.3) gives a lower RMSE (6.49 km) but also a lower-variance target (mean-baseline RMSE 7.70 vs. 8.96 km), a similar correlation (0.54) and a slightly lower $R^2$ (0.29); 23% of its predictions fall within ±0.5 km of 7.00 km. The filtered results are therefore not artificially favourable, but they describe the ~81% of events with a non-default depth.

**8. Overfitting check.** Random Forest has the largest train–test gap (magnitude RMSE 0.231 → 0.294; depth 5.64 → 7.53 km), while Gradient Boosting is more balanced (0.272 → 0.294; 6.56 → 7.51 km) with statistically indistinguishable test scores. It is also more than ten times faster to train on this machine (≈ 1.2 s vs. ≈ 16 s for magnitude; Section 6.3). Gradient Boosting is therefore the preferred model.

**Conclusion.** Both numeric targets are learnable above the mean baseline, but only partially ($R^2 \approx 0.50$ for magnitude and $\approx 0.30$ for depth), and half of the magnitude signal comes from catalogue metadata (`type_*`). Better results would require physics-based features such as distance to the nearest mapped fault, time since the last large mainshock (Omori–Utsu decay) or local event density.